# Behavior sanity check

Histograms of behavior values along each dimension, for each session separately.

In [1]:
import sys
sys.path.append('/home/adam/digitalbat')

import numpy as np
import torch
from matplotlib import pyplot as plt

from digitalbat.datasets.data import bat_data_loader


ModuleNotFoundError: No module named 'digitalbat'

In [ ]:
data_config = dict(
    use_speed_head_dir=False,
    use_IMU_flag=False,
    use_click_flag=False,
    use_LFP_flag=False,
    use_ripple_flag=False,
    only_flight_flag=True,
    no_3d_location=False,
    spike_count_flag=True,
    bin_width=0.10,
    split_flights_for_train="random",
    type_train_num=1,
    type_valid_num=1,
    special_id_count_thresh=3,
    only_use_special_id_4train_flag=False,
    shuffle_flag=True,
    time_shuffle_for_trajecotry_flag=False,
    simulate_place_cell_flag=False,
    split_each_trajectory_flag=True,
    window_size=None,
    stride_train=None,
    stride_test=None,
    batch_size_train=None,
    batch_size_test=None,
    session_names=[
        "Dataset_3_14543_240419",
        "Dataset_3_14543_240420",
        "Dataset_3_14543_240421",
        "Dataset_3_29959_240402",
        "Dataset_3_29959_240403",
        "Dataset_3_29959_240404",
        "Dataset_2_14445_231208",
        "Dataset_2_14445_231209",
        "Dataset_1_32622_231009",
        "Dataset_1_32622_231010",
    ],
)
dataloaders, behavior_dim, unit_num_dict, meta_info_dict = bat_data_loader(**data_config)
session_names = list(dataloaders["train"].keys())
print(f"behavior_dim: {behavior_dim}")
print(f"sessions ({len(session_names)}):")
for s in session_names:
    print(f"  {s}")


In [ ]:
# Collect masked behavior values per session, concatenated across train+valid+test.
behavior_by_session = {}
for session_name in session_names:
    parts = []
    for tier in ['train', 'valid', 'test']:
        for batch in dataloaders[tier][session_name]:
            X = batch[0]      # (N, T, behavior_dim)
            mask = batch[2]   # (N, T)
            parts.append(X[mask == 1.0].cpu().numpy())  # (M, behavior_dim)
    behavior_by_session[session_name] = np.concatenate(parts, axis=0)

for s, arr in behavior_by_session.items():
    print(f'{s}: shape={arr.shape}, '
          f'mins={arr.min(axis=0).round(2)}, maxs={arr.max(axis=0).round(2)}')


In [2]:
# Histograms: rows = sessions, cols = behavior dims
dim_names = ['x (z-score)', 'y (z-score)', 'z (z-score)'] + [f'dim{i}' for i in range(3, behavior_dim)]
n_rows = len(session_names)
n_cols = behavior_dim
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 2 * n_rows), squeeze=False)
for r, session_name in enumerate(session_names):
    arr = behavior_by_session[session_name]
    for c in range(n_cols):
        ax = axes[r, c]
        ax.hist(arr[:, c], bins=60, color='steelblue', edgecolor='none')
        if r == 0:
            ax.set_title(dim_names[c])
        if c == 0:
            ax.set_ylabel(session_name, fontsize=8)
        ax.tick_params(labelsize=7)
plt.tight_layout()
plt.show()


NameError: name 'behavior_dim' is not defined